In [ ]:
from nodes.nodes import node_classificador, run_geo_reasoning, tool_node, node_triagem, router_triagem, node_normalizacao, router_geo, node_validador, router_validacao_e_check, node_triagem_comentario, node_classificador_comentario
from langgraph.graph import StateGraph, END
from classes.classes import GraphState, GraphStateComentario, EventoEnriquecido
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
from tqdm import tqdm
import json
import os
from sqlalchemy import create_engine
import time
import ast
from langfuse.langchain import CallbackHandler


langfuse_handler = CallbackHandler()

In [ ]:
# 1. Cria o Grafo
workflow = StateGraph(GraphState)

# 2. Adiciona os Nós
workflow.add_node("triagem", node_triagem)
workflow.add_node("extrator", node_classificador)      
workflow.add_node("validador", node_validador)         
workflow.add_node("geo_agent", run_geo_reasoning)      
workflow.add_node("tools_geo", tool_node)    
workflow.add_node("normalizacao", node_normalizacao)   

# 3. Define a Entrada
workflow.set_entry_point("triagem")

# 4. Triagem -> Extrator ou Fim
workflow.add_conditional_edges(
    "triagem",
    router_triagem,
    {
        "extrator": "extrator",
        END: END
    }
)

# 5. Extrator -> Validador (Fluxo Obrigatório: Quem extrai, tem que validar)
workflow.add_edge("extrator", "validador") 

# 6. Validador -> (Decisão Tripla: Corrigir / Fim / Geo)
workflow.add_conditional_edges(
    "validador",
    router_validacao_e_check, # <--- AQUI ESTÁ A MUDANÇA
    {
        "retry": "extrator",           # Erro: Volta e tenta de novo
        "normalizacao": "normalizacao",# Vazio: Vai pro fim
        "geo_agent": "geo_agent"       # Ok: Vai pro mapa
    }
)

# 7. O Loop do Agente Geográfico
workflow.add_conditional_edges(
    "geo_agent",
    router_geo,
    {
        "tools_geo": "tools_geo",       # Usar ferramenta
        "normalizacao": "normalizacao", # Finalizar trabalho
        "geo_agent": "geo_agent"        # Pensar de novo (raro, mas possível)
    }
)

# 8. Retorno da Ferramenta pro Agente
workflow.add_edge("tools_geo", "geo_agent")

# 9. Finalização
workflow.add_edge("normalizacao", END)

# Compila
app = workflow.compile()
app

## Testes com modelos locais

In [ ]:
DB_NAME=os.getenv('DB_NAME')
DB_USER=os.getenv('DB_USER')
DB_PW=os.getenv('DB_PW')
DB_HOST=os.getenv('DB_HOST')
DB_PORT=os.getenv('DB_PORT')
engine = create_engine(f"postgresql://{DB_USER}:{DB_PW}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

app.invoke(input={'texto_input': """oi"""},
config={'callbacks':[langfuse_handler]})

## Extração das publicações

In [ ]:
DB_NAME=os.getenv('DB_NAME')
DB_USER=os.getenv('DB_USER')
DB_PW=os.getenv('DB_PW')
DB_HOST=os.getenv('DB_HOST')
DB_PORT=os.getenv('DB_PORT')
engine = create_engine(f"postgresql://{DB_USER}:{DB_PW}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

df_publicacoes = pd.read_sql("""
select id_publicacao, texto_publicacao
from tcc.publicacao p
left join tcc.fato_publicacao_percepcao fpp
on p.id_publicacao = fpp.publicacao_id_publicacao""", con=engine)
df_publicacoes

In [ ]:
app.invoke(input={'texto_input': """VIROU NOVELA DAS 8 DA GLOBO! Na manhã desta quinta-feira, a Polícia Civil esteve na residência do cantor MC Poze do Rodo, no Rio de Janeiro, para intimar sua esposa, Viviane Noronha, a prestar depoimento. A ação ocorre após Viviane afirmar em seu perfil no Instagram que policiais teriam roubado joias do funkeiro durante uma operação anterior realizada na casa do casal.

A declaração gerou repercussão nas redes sociais e chamou a atenção do chefe de Polícia, que agora investiga a veracidade da acusação. A intimação tem como objetivo esclarecer os fatos e apurar se houve crime por parte dos agentes ou possível calúnia.

Nas redes sociais, MC Poze se manifestou com indignação. Em uma publicação nos stories, ele criticou a ação da polícia, pedindo que foquem em prender criminosos de verdade e deixem sua família em paz: "Vão prender bandido e deixem minha família quieta", escreveu o artista.

Até o momento, a Polícia Civil não divulgou mais detalhes sobre o andamento da investigação.

— Roubou mesmo????"""})

In [ ]:
resultado = app.invoke(
    input={"texto_input": "Traficantes cobram taxa de segurança de vendedores em frente à unirio da urca"},
    config={'callbacks':[langfuse_handler]}
)

print(resultado)


In [ ]:
ids = pd.read_sql("select * from tcc.fato_publicacao_percepcao where autor_crime_id_autor = 15", con=engine)['publicacao_id_publicacao']
df_publicacoes = df_publicacoes[df_publicacoes['id_publicacao'].isin(ids)]
df_publicacoes

In [ ]:
# --- CONFIGURAÇÃO INICIAL ---
ARQUIVO_SAIDA = "resultados/resultados_processamento_posts_tcc.jsonl"

# Função auxiliar (serialização)
def serializar_evento(evento: EventoEnriquecido) -> dict:
    return {
        'autor_crime_id_autor': evento.autor.value,
        'bairro_id_bairro': evento.bairro_id,
        'raciocinio': evento.raciocinio,
        'logradouro': evento.logradouro,
        'nome_grupo': evento.grupo,
        'crime_id_crime': evento.crime.value,
        'local_texto_original': evento.local,
        'id_evento_llm': evento.id
    }

# --- LÓGICA DE RETOMADA (CHECKPOINT) ---
ids_processados = set()

if os.path.exists(ARQUIVO_SAIDA):
    print(f"Arquivo '{ARQUIVO_SAIDA}' encontrado. Mapeando registros já processados...")
    try:
        with open(ARQUIVO_SAIDA, 'r', encoding='utf-8') as f:
            for linha in f:
                linha = linha.strip()
                if not linha: continue # Pula linhas vazias
                try:
                    registro = json.loads(linha)
                    # Adiciona ao set para busca rápida O(1)
                    ids_processados.add(registro['id_publicacao'])
                except json.JSONDecodeError:
                    print("Aviso: Linha corrompida encontrada e ignorada no arquivo JSONL.")
    except Exception as e:
        print(f"Erro ao ler arquivo de checkpoint: {e}")

print(f"Registros já salvos: {len(ids_processados)}")

# Filtra o DataFrame para pegar apenas o que FALTA
df_para_processar = df_publicacoes[~df_publicacoes['id_publicacao'].isin(ids_processados)]

print(f"Total original: {len(df_publicacoes)}")
print(f"Faltam processar: {len(df_para_processar)}")

if len(df_para_processar) == 0:
    print("Todos os registros já foram processados!")
else:
    print("Iniciando processamento incremental...")

    # --- LOOP DE PROCESSAMENTO (Usando o DF filtrado) ---
    for index, row in tqdm(df_para_processar.iterrows(), total=df_para_processar.shape[0]):
        id_pub = row['id_publicacao']
        texto = row['texto_publicacao']
        
        # Estrutura base
        registro_final = {
            "id_publicacao": id_pub,
            "erro_validacao": None,
            "classificacao_relevancia": False,
            "eventos": [] 
        }
        
        try:
            # 1. Invoca o LangGraph (com limite de recursão aumentado)
            output = app.invoke(
                {"texto_input": texto},
                config={"recursion_limit": 100, 'callbacks':[langfuse_handler]} 
            )
            
            # 2. Extrai metadados
            registro_final["erro_validacao"] = output.get("erro_validacao")
            registro_final["classificacao_relevancia"] = output.get("eh_relevante", False)
            
            # 3. Processa a lista de eventos
            eventos_objetos = output.get("eventos_finais", [])
            
            if eventos_objetos:
                lista_dicts = [serializar_evento(evt) for evt in eventos_objetos]
                registro_final["eventos"] = lista_dicts

        except Exception as e:
            # Captura erro crítico
            erro_msg = f"CRASH SYSTEM: {str(e)}"
            print(f"Erro no ID {id_pub}: {erro_msg}")
            print(f"Publicação: ", texto)
            registro_final["erro_validacao"] = erro_msg

        # 4. SALVAMENTO IMEDIATO
        with open(ARQUIVO_SAIDA, 'a', encoding='utf-8') as f:
            linha_json = json.dumps(registro_final, ensure_ascii=False)
            f.write(linha_json + "\n")
        
        # Pequena pausa para não sobrecarregar I/O ou API se necessário
        time.sleep(0.3) 

    print("Processamento finalizado.")

In [ ]:
posts_com_eventos = []
try:
    with open(ARQUIVO_SAIDA, 'r', encoding='utf-8') as f:
        for linha in f:
            # Carrega a linha atual como um dicionário
            dados = json.loads(linha)
            
            # Verifica se a chave 'eventos' existe e se NÃO está vazia
            if dados.get('eventos') and len(dados['eventos']) > 0:
                posts_com_eventos.append(dados)

    print(f"Total de registros com crimes encontrados: {len(posts_com_eventos)}")
    
    # Exemplo do primeiro registro encontrado para conferência
    if posts_com_eventos:
        print(json.dumps(posts_com_eventos[0], indent=2, ensure_ascii=False))

except FileNotFoundError:
    print(f"Erro: O arquivo '{ARQUIVO_SAIDA}' não foi encontrado.")
except Exception as e:
    print(f"Ocorreu um erro ao ler o arquivo: {e}")

In [ ]:
df_bruto = pd.DataFrame(posts_com_eventos)
df_explodido = df_bruto.explode('eventos').reset_index(drop=True)
df_explodido

In [ ]:
df_eventos_cols = pd.json_normalize(df_explodido['eventos'])
df_final = pd.concat([
    df_explodido[['id_publicacao']], # Colunas do post
    df_eventos_cols # Colunas do evento explodido
], axis=1).drop(columns=['id_evento_llm', 'local_texto_original']).rename(columns={'id_publicacao': 'publicacao_id_publicacao'})
df_final = df_final[df_final['crime_id_crime'] != 42]
df_final

In [ ]:
df_final.to_sql("fato_publicacao_percepcao", schema='tcc', if_exists='append', con=engine, method='multi', index=False)

## Extração dos comentários

In [ ]:
# 1. Cria o Grafo
workflow = StateGraph(GraphStateComentario)

# 2. Adiciona os Nós
workflow.add_node("triagem", node_triagem_comentario)
workflow.add_node("extrator", node_classificador_comentario)      
workflow.add_node("validador", node_validador)         
workflow.add_node("geo_agent", run_geo_reasoning)      
workflow.add_node("tools_geo", tool_node)    
workflow.add_node("normalizacao", node_normalizacao)   

# 3. Define a Entrada
workflow.set_entry_point("triagem")

# 4. Triagem -> Extrator ou Fim
workflow.add_conditional_edges(
    "triagem",
    router_triagem,
    {
        "extrator": "extrator",
        END: END
    }
)

# 5. Extrator -> Validador (Fluxo Obrigatório: Quem extrai, tem que validar)
workflow.add_edge("extrator", "validador") 

# 6. Validador -> (Decisão Tripla: Corrigir / Fim / Geo)
workflow.add_conditional_edges(
    "validador",
    router_validacao_e_check,
    {
        "retry": "extrator",           # Erro: Volta e tenta de novo
        "normalizacao": "normalizacao",# Vazio: Vai pro fim
        "geo_agent": "geo_agent"       # Ok: Vai pro mapa
    }
)

# 7. O Loop do Agente Geográfico
workflow.add_conditional_edges(
    "geo_agent",
    router_geo,
    {
        "tools_geo": "tools_geo",       # Usar ferramenta
        "normalizacao": "normalizacao", # Finalizar trabalho
        "geo_agent": "geo_agent"        # Pensar de novo (raro, mas possível)
    }
)

# 8. Retorno da Ferramenta pro Agente
workflow.add_edge("tools_geo", "geo_agent")

# 9. Finalização
workflow.add_edge("normalizacao", END)

# Compila
app = workflow.compile()
app

In [ ]:
input_comentario_relevante = {
    "texto_input": "Foi horrível! Eles desceram a ladeira atirando e entraram na Barata Ribeiro.",
    "contexto_local": "Copacabana",
    "contexto_autores": "Traficantes do CV",
    "contexto_crimes": "Tiroteio",
}

app.invoke(input_comentario_relevante, config={'callbacks': [langfuse_handler]})

In [ ]:
query_comentarios_crimes = """
WITH publicacao_crime AS (
    SELECT 
        p.id_publicacao,
        STRING_AGG(DISTINCT ac.nome_autor, ', ') AS lista_autores,
        STRING_AGG(DISTINCT c.nome_crime, ', ') AS lista_crimes,
        STRING_AGG(distinct b.nome_bairro, ', ') as lista_bairros
    FROM tcc.fato_publicacao_percepcao fpp
    LEFT JOIN tcc.autor_crime ac
        ON fpp.autor_crime_id_autor = ac.id_autor
    LEFT JOIN tcc.bairro b
        ON fpp.bairro_id_bairro = b.id_bairro
    LEFT JOIN tcc.crime c
        ON fpp.crime_id_crime = c.id_crime
    LEFT JOIN tcc.publicacao p
        ON fpp.publicacao_id_publicacao = p.id_publicacao
    WHERE c.nome_crime != 'Sem Crime' and b.nome_bairro != 'Não Identificado'
    GROUP BY p.id_publicacao
)
SELECT 
    pc.*, 
    c.id_comentario,
    c.texto_comentario
FROM publicacao_crime pc
LEFT JOIN tcc.comentario c
    ON pc.id_publicacao = c.publicacao_id_publicacao
WHERE
    c.texto_comentario ~* '(mat(ou|ar|am)|assassin|execut|bale(a|o)|atir|dispar|espanc|agred|feri|roub|furt|assalt|subtra|lev(ou|aram)|estupr|violent|sequestr|extorqu|invad|tom(ou|aram)|trafic|incendi|queim)';
"""

df_comentarios_selecionados = pd.read_sql(query_comentarios_crimes, con=engine)
df_comentarios_selecionados

In [ ]:
ids = pd.read_sql("select * from tcc.fato_comentario_percepcao where autor_crime_id_autor = 15", con=engine)['comentario_id_comentario']
df_comentarios_selecionados = df_comentarios_selecionados[df_comentarios_selecionados['id_comentario'].isin(ids)]

In [ ]:
ARQUIVO_SAIDA_COMENTARIOS = "resultados/resultados_processamento_comentarios_tcc.jsonl"

def serializar_evento_comentario(evento) -> dict:
    """Extrai os dados do objeto EventoEnriquecido, pegando valores de Enums."""
    return {
        'autor_crime_id_autor': evento.autor.value if hasattr(evento.autor, 'value') else evento.autor,
        'bairro_id_bairro': evento.bairro_id,
        'raciocinio': evento.raciocinio,
        'logradouro': evento.logradouro,
        'nome_grupo': evento.grupo,
        'crime_id_crime': evento.crime.value if hasattr(evento.crime, 'value') else evento.crime,
        'id_evento_llm': evento.id
    }

In [ ]:
ids_comentarios_processados = set()

# Checkpoint
if os.path.exists(ARQUIVO_SAIDA_COMENTARIOS):
    with open(ARQUIVO_SAIDA_COMENTARIOS, 'r', encoding='utf-8') as f:
        for linha in f:
            try:
                reg = json.loads(linha)
                ids_comentarios_processados.add(reg['id_comentario'])
            except: continue

# Filtra o DataFrame (df_comentarios é o seu DF com as 6 colunas citadas)
df_para_processar = df_comentarios_selecionados[~df_comentarios_selecionados['id_comentario'].isin(ids_comentarios_processados)]

print(f"Comentários para processar: {len(df_para_processar)}")

if len(df_para_processar) > 0:
    for index, row in tqdm(df_para_processar.iterrows(), total=df_para_processar.shape[0]):
        
        # Monta o input para o GraphStateComentario
        # Nota: Ajustamos para garantir que pegamos o texto e os contextos herdados
        input_langgraph = {
            "texto_input": str(row['texto_comentario']),
            "contexto_local": str(row['lista_bairros']),
            "contexto_autores": str(row['lista_autores']),
            "contexto_crimes": str(row['lista_crimes']),
            "messages": [] # Inicializa lista de mensagens para o MessagesState
        }
        
        registro_final = {
            "id_comentario": row['id_comentario'],
            "id_publicacao_pai": row['id_publicacao'],
            "classificacao_relevancia": False,
            "eventos": [],
            "erro": None
        }
        
        try:
            # Invoca o Grafo
            output = app.invoke(input_langgraph, config={'callbacks': [langfuse_handler]})
            
            registro_final["classificacao_relevancia"] = output.get("eh_relevante", False)
            
            # Se houver novos eventos/detalhes extraídos do comentário
            eventos_objetos = output.get("eventos_finais", [])
            if eventos_objetos:
                registro_final["eventos"] = [serializar_evento_comentario(evt) for evt in eventos_objetos]
                
        except Exception as e:
            registro_final["erro"] = str(e)
            print(f"Erro no comentário {row['id_comentario']}: {e}")

        # Salvamento incremental (Append)
        with open(ARQUIVO_SAIDA_COMENTARIOS, 'a', encoding='utf-8') as f:
            f.write(json.dumps(registro_final, ensure_ascii=False) + "\n")
            
        time.sleep(0.2) # Respiro para a API

In [ ]:
dados_para_dataframe = []
if os.path.exists(ARQUIVO_SAIDA_COMENTARIOS):
    with open(ARQUIVO_SAIDA_COMENTARIOS, 'r', encoding='utf-8') as f:
        for linha in f:
            linha = linha.strip()
            if not linha: continue
            
            try:
                registro = json.loads(linha)
                eventos = registro.get("eventos_finais") or registro.get("eventos") or []
                
                if eventos:                    
                    # Itera sobre cada evento encontrado no comentário
                    for evt in eventos:                        
                        # Achata o dicionário (Flat) para o Pandas
                        item = {
                            "comentario_id_comentario": registro.get("id_comentario"),
                            "raciocinio": evt.get("raciocinio"),
                            "crime_id_crime": evt.get("crime_id_crime"),
                            "autor_crime_id_autor": evt.get("autor_crime_id_autor"),
                            "bairro_id_bairro": int(evt.get("bairro_id_bairro")),
                            "logradouro": None if evt.get("logradouro") == 'None' else evt.get("logradouro"),
                            "nome_grupo": evt.get("nome_grupo")
                        }
                        dados_para_dataframe.append(item)
                    
            except json.JSONDecodeError:
                print("Linha corrompida ignorada.")
df_comentario = pd.DataFrame(dados_para_dataframe)
df_comentario = df_comentario[df_comentario['crime_id_crime'] != 42]
df_comentario

In [ ]:
df_comentario.to_sql("fato_comentario_percepcao", schema='tcc', index=False, if_exists='append', con=engine, method='multi')